# 02 - Borrower risk segmentation

Rule-based segments (`sql/marts/07_borrower_mart.sql`) versus K-Means on behavioural features (`credit_risk.segmentation.clustering`).

In [1]:
import duckdb
import pandas as pd

from credit_risk.config import get_settings

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.4f}".format)
settings = get_settings()
con = duckdb.connect(str(settings.paths.warehouse), read_only=True)
q = lambda sql: con.execute(sql).df()  # noqa: E731

In [2]:
bm = q("SELECT * FROM marts.borrower_mart")
bm["risk_segment"].value_counts()

risk_segment
Emerging Delinquency                  1453
Low Risk / Strong Repayer             1047
Default / Recovery                    1024
Moderate Risk                          691
Low Risk / New Borrower                555
High Risk / Persistent Delinquency     205
Name: count, dtype: int64

## Segment outcomes
Descriptive only: segments are assigned with hindsight.

In [3]:
q((settings.paths.sql / "analytics/08_default_rate_by_segment.sql").read_text())

,risk_segment,borrowers,loans,bad_30dpd_rate,default_rate,outstanding_principal
0,Low Risk / New Borrower,555,616,0.0000,0.0000,"1,671,161.0000"
1,Low Risk / Strong Repayer,1047,5290,0.0000,0.0000,"5,603,536.0000"
2,Moderate Risk,691,2398,0.0129,0.0000,"1,917,047.0000"
3,Emerging Delinquency,1453,3784,0.1203,0.0000,"1,837,719.0000"
4,High Risk / Persistent Delinquency,205,585,0.4231,0.0000,"1,284,400.0000"
5,Default / Recovery,1024,2447,0.4484,0.4243,"1,530,157.0000"


## Rules vs clusters

In [4]:
clusters = q("SELECT * FROM marts.borrower_clusters")
pd.crosstab(clusters["risk_segment"], clusters["cluster"])

cluster,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6
risk_segment,,,,,,
Default / Recovery,0,0,12,272,212,528
Emerging Delinquency,105,631,260,386,69,2
High Risk / Persistent Delinquency,0,0,0,28,177,0
Low Risk / New Borrower,474,0,81,0,0,0
Low Risk / Strong Repayer,616,0,431,0,0,0
Moderate Risk,102,339,231,19,0,0


In [5]:
from sklearn.metrics import adjusted_rand_score

adjusted_rand_score(clusters["risk_segment"], clusters["cluster"])

0.27640278199121565

**Reading.** Moderate agreement: K-Means separates borrowers by engagement (tenure, frequency) as well as delinquency, whereas the rules are driven by severity and recency thresholds. The full discussion is in `reports/segmentation_report.md`.